# House Prices | XGB+Cat+LGB etc. | LB 0.12111

Predicting house prices on the classic Ames, Iowa dataset. The notebook is written so you can read top-to-bottom — each section briefly explains the decision behind the code, then runs it.

## Our key contribution

Most public 0.12-range notebooks rely on either heavy stacking (Serigne-style meta-learner) or a single-seed boosting blend. On a small dataset (1,458 rows × ~250 features), these are prone to fold-level variance that doesn't generalize cleanly to the leaderboard. Our insight is to **average each tree model across 3 random seeds** (XGBoost, CatBoost, LightGBM) and then **hedge-blend** the resulting 5-way ensemble with a previously-validated submission — averaging two different preprocessing pipelines.

This took our LB through `0.12247 → 0.12142 → 0.12111` across three iterations. The final improvement came from adding LightGBM as a genuinely diverse signal (its predictions correlate with XGB/CatBoost at ~0.93, not the ~0.99 you typically see between variants of the same boosting method).

The `D_safe_lin` blend weights (Lasso 0.35, Ridge 0.10, XGB 0.20, CatBoost 0.20, LGB 0.15) were selected via OOF from 5 candidate schemes. Linear weight was consistently the strongest single contributor — a useful reminder that regularized linear models remain very competitive on small tabular regression.

## What this notebook does

- Reads competition train/test (1,460 + 1,459 rows, 80 features).
- Removes 2 documented outliers.
- Imputes missing values semantically (most NAs mean "feature absent", not "unknown").
- Encodes quality ratings manually so the order is preserved.
- Engineers 14 new features (totals, ratios, presence flags, quality × area, neighborhood target encoding).
- Corrects skewness and one-hot-encodes categoricals (~265 columns total).
- Trains 5 models with 5-fold OOF: Lasso, Ridge, XGBoost (GPU, 3-seed), CatBoost (GPU, 3-seed), LightGBM (CPU, 3-seed).
- Blends with weights chosen by OOF and writes the submission.

## 0. Setup

Pin the random seed for reproducibility, set up a clean plot style, and auto-detect the data path (works on both Kaggle and a local machine).

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from scipy import stats
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.base import clone
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import warnings; warnings.filterwarnings('ignore')

# Plot style
PALETTE = ['#2C3E50', '#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA', 'axes.facecolor': '#FAFAFA',
    'axes.edgecolor': '#CCCCCC', 'axes.grid': True,
    'grid.color': '#E8E8E8', 'grid.linewidth': 0.7,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'legend.frameon': False,
})

# Auto-detect data path — Kaggle mounts competitions in two possible patterns
CANDIDATE_DIRS = [
    '/kaggle/input/house-prices-advanced-regression-techniques',
    '/kaggle/input/competitions/house-prices-advanced-regression-techniques',
    '../data',
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(f'{d}/train.csv')), None)
if DATA_DIR is None:
    raise FileNotFoundError(f'train.csv not found in any of: {CANDIDATE_DIRS}')
OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

SEED = 42
N_FOLDS = 5
SEEDS = [42, 2023, 7]
np.random.seed(SEED)
t0 = time.time()
print(f'data: {DATA_DIR}  | out: {OUT_DIR}')

## 1. Load data & remove documented outliers

The original Ames dataset author (Dean De Cock) flagged a handful of houses that are very large (`GrLivArea > 4000`) but sold for very little (`< $300K`) — almost certainly non-market sales. Removing them helps the model learn the normal market.

We also `log1p` the target. The competition metric is RMSE on `log(SalePrice)`, so training on log-prices aligns the loss with the score and stabilizes residuals across the price range.

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'train: {train.shape}  | test: {test.shape}')

# Keep raw copy for later visualizations
train_raw = train.copy()

mask = (train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)
print(f'Removing {mask.sum()} outliers')
train = train.loc[~mask].reset_index(drop=True)

y = np.log1p(train['SalePrice']).values
test_ids = test['Id']
ntrain = len(train)

# Neighborhood target encoding (train-only stats)
nbhd_median = train.groupby('Neighborhood')['SalePrice'].median()

all_data = pd.concat([train.drop(columns=['Id','SalePrice']),
                      test.drop(columns=['Id'])], ignore_index=True)
print(f'combined: {all_data.shape}')

### Visualization — Target distribution & documented outliers

The raw `SalePrice` is heavily right-skewed; `log1p` makes it close to normal. The 2 red points (large area + low price) are flagged by the original Ames dataset author as non-market sales — we remove them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('SalePrice — distribution & outliers', fontsize=14, fontweight='bold')

axes[0].hist(train_raw['SalePrice'] / 1000, bins=50, color=PALETTE[0], edgecolor='white', alpha=0.85)
axes[0].axvline(train_raw['SalePrice'].median() / 1000, color=PALETTE[1], lw=2, ls='--',
                label=f'Median ${train_raw["SalePrice"].median()/1000:.0f}K')
axes[0].set_title(f'Raw SalePrice (skew={train_raw["SalePrice"].skew():.2f})')
axes[0].set_xlabel('SalePrice ($K)'); axes[0].legend()

log_y = np.log1p(train_raw['SalePrice'])
axes[1].hist(log_y, bins=50, color=PALETTE[2], edgecolor='white', alpha=0.85)
axes[1].set_title(f'log1p(SalePrice) (skew={log_y.skew():.2f}) — close to normal')
axes[1].set_xlabel('log1p(SalePrice)')

mask_out = (train_raw['GrLivArea'] > 4000) & (train_raw['SalePrice'] < 300000)
axes[2].scatter(train_raw.loc[~mask_out, 'GrLivArea'], train_raw.loc[~mask_out, 'SalePrice'] / 1000,
                alpha=0.4, s=18, color=PALETTE[0], label='Kept')
axes[2].scatter(train_raw.loc[mask_out, 'GrLivArea'], train_raw.loc[mask_out, 'SalePrice'] / 1000,
                color=PALETTE[1], s=120, label='Removed', edgecolor='black', linewidth=1.5)
axes[2].set_xlabel('GrLivArea (sqft)'); axes[2].set_ylabel('SalePrice ($K)')
axes[2].set_title('Outlier mask: GrLivArea>4000 & SalePrice<$300K')
axes[2].legend()

plt.tight_layout(); plt.show()

## 2. Missing values — "absence has meaning"

A key insight in this dataset: many NA values don't mean *unknown*, they mean *the feature isn't there*. `PoolQC = NA` means the house has no pool, not that the pool quality is missing. Same for `Alley`, `Fence`, `GarageType`, `BsmtQual`, etc.

So we fill these with `'None'` rather than imputing. For genuinely missing categoricals we use the mode; `LotFrontage` is filled with the median for the same `Neighborhood` (houses on the same block tend to have similar frontage).

In [ ]:
# NA = 'None' (feature does not exist)
for col in ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu','GarageType',
            'GarageFinish','GarageQual','GarageCond','BsmtQual','BsmtCond',
            'BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']:
    all_data[col] = all_data[col].fillna('None')
all_data['MSSubClass'] = all_data['MSSubClass'].fillna('None').astype(str)

# NA = 0 (no area / no count)
for col in ['GarageYrBlt','GarageArea','GarageCars','BsmtFinSF1','BsmtFinSF2',
            'BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath','MasVnrArea']:
    all_data[col] = all_data[col].fillna(0)

# LotFrontage: same neighborhood = similar frontage
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda s: s.fillna(s.median()))

# Remaining categoricals: mode
for col in ['MSZoning','Electrical','KitchenQual','Exterior1st','Exterior2nd',
            'SaleType','Functional','Utilities']:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print(f'Remaining NA: {all_data.isna().sum().sum()}')

## 3. Ordinal encoding (Poor → Excellent)

Quality columns have a natural order: `Po < Fa < TA < Gd < Ex`. The naïve `LabelEncoder` sorts alphabetically (`Ex=0, Fa=1, Gd=2, None=3, Po=4, TA=5`), which destroys that order. A manual mapping `{'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5}` preserves it, letting both linear and tree models use the rank directly.

In [ ]:
qual_map = {'Po':1,'Fa':2,'TA':3,'Gd':4,'Ex':5,'None':0,'NA':0}
for col in ['ExterQual','ExterCond','BsmtQual','BsmtCond','HeatingQC',
            'KitchenQual','FireplaceQu','GarageQual','GarageCond','PoolQC']:
    all_data[col] = all_data[col].map(qual_map).fillna(0).astype(int)

all_data['Functional']   = all_data['Functional'].map(
    {'Sal':1,'Sev':2,'Maj2':3,'Maj1':4,'Mod':5,'Min2':6,'Min1':7,'Typ':8}).fillna(0).astype(int)
all_data['BsmtFinType1'] = all_data['BsmtFinType1'].map(
    {'NA':0,'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6}).fillna(0).astype(int)
all_data['BsmtFinType2'] = all_data['BsmtFinType2'].map(
    {'NA':0,'None':0,'Unf':1,'LwQ':2,'Rec':3,'BLQ':4,'ALQ':5,'GLQ':6}).fillna(0).astype(int)
all_data['BsmtExposure'] = all_data['BsmtExposure'].map(
    {'NA':0,'None':0,'No':1,'Mn':2,'Av':3,'Gd':4}).fillna(0).astype(int)
all_data['GarageFinish'] = all_data['GarageFinish'].map(
    {'NA':0,'None':0,'Unf':1,'RFn':2,'Fin':3}).fillna(0).astype(int)
all_data['PavedDrive']   = all_data['PavedDrive'].map({'N':0,'P':1,'Y':2}).fillna(0).astype(int)
print('Ordinal encoding done')

## 4. Feature engineering — add signal the model can't easily derive

A few hand-crafted features that mirror how buyers actually think:

- **Totals**: `TotalSF` = basement + 1st + 2nd, `TotalBathrooms`, `TotalPorchSF`, `AllFloorsSF`.
- **Age**: `HouseAge` and `YearsSinceRemod` capture recency.
- **Presence flags**: `HasPool`, `HasGarage`, `Has2ndFloor`, `HasBsmt`, `HasFireplace` — a clean 0/1 even when the area column is 0.
- **Quality × area interactions**: `QualArea = OverallQual * GrLivArea`, `QualTotalSF` — "luxury sqft" is worth more than basic sqft.
- **Neighborhood target encoding**: replace the categorical name with the median SalePrice from the training set.

In [ ]:
# Aggregate footprints
all_data['TotalSF']        = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBathrooms'] = (all_data['FullBath'] + 0.5*all_data['HalfBath']
                              + all_data['BsmtFullBath'] + 0.5*all_data['BsmtHalfBath'])
all_data['TotalPorchSF']   = (all_data['OpenPorchSF'] + all_data['EnclosedPorch']
                              + all_data['3SsnPorch'] + all_data['ScreenPorch'])
all_data['AllFloorsSF']    = all_data['1stFlrSF'] + all_data['2ndFlrSF']

# Age and remodeling
all_data['HouseAge']        = all_data['YrSold'] - all_data['YearBuilt']
all_data['YearsSinceRemod'] = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['IsRemodeled']     = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)
all_data['IsNewHouse']      = (all_data['YrSold'] == all_data['YearBuilt']).astype(int)

# Presence flags
all_data['HasPool']      = (all_data['PoolArea']    > 0).astype(int)
all_data['HasGarage']    = (all_data['GarageArea']  > 0).astype(int)
all_data['Has2ndFloor']  = (all_data['2ndFlrSF']    > 0).astype(int)
all_data['HasBsmt']      = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces']  > 0).astype(int)

# Quality × area interactions
all_data['QualArea']    = all_data['OverallQual'] * all_data['GrLivArea']
all_data['QualTotalSF'] = all_data['OverallQual'] * all_data['TotalSF']

# Neighborhood target encoding
all_data['NeighborhoodPrice'] = all_data['Neighborhood'].map(nbhd_median).fillna(nbhd_median.median())

print(f'After feature engineering: {all_data.shape}')

### Visualization — Top 20 feature correlations

Red bars are features we engineered. The fact that `QualArea`, `QualTotalSF`, `TotalSF`, and `NeighborhoodPrice` all rank near the top validates that the engineered features carry real signal.

In [ ]:
train_with_y = all_data.iloc[:ntrain].copy()
train_with_y['log_SalePrice'] = y
numeric = train_with_y.select_dtypes(include=[np.number])
corr = numeric.corrwith(train_with_y['log_SalePrice']).abs().sort_values(ascending=False)
top20 = corr.drop('log_SalePrice').head(20)

engineered = {
    'TotalSF','TotalBathrooms','TotalPorchSF','AllFloorsSF',
    'HouseAge','YearsSinceRemod','IsRemodeled','IsNewHouse',
    'HasPool','HasGarage','Has2ndFloor','HasBsmt','HasFireplace',
    'QualArea','QualTotalSF','NeighborhoodPrice',
}
colors = [PALETTE[1] if f in engineered else PALETTE[0] for f in top20.index]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top20)), top20.values, color=colors, alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20.index)
ax.invert_yaxis()
ax.set_xlabel('|correlation| with log(SalePrice)')
ax.set_title('Top 20 features by absolute correlation — RED = engineered')
for i, v in enumerate(top20.values):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

n_eng = sum(1 for f in top20.index if f in engineered)
print(f'Engineered features in top 20: {n_eng} / 20')

## 5. Fix skewness, then one-hot encode

Many area/size columns are right-skewed (most houses are average, a few are huge). Linear models prefer roughly symmetric distributions, so we apply `log1p` wherever `|skew| > 0.75`. Then we one-hot encode the remaining categoricals — the final design matrix has around 265 columns.

In [ ]:
numeric_feats = all_data.select_dtypes(include=[np.number]).columns
skewness = all_data[numeric_feats].apply(lambda x: skew(x.dropna()))
skewed = skewness[skewness.abs() > 0.75].index.tolist()
for col in skewed:
    all_data[col] = np.log1p(all_data[col].clip(lower=0))
print(f'Log1p applied to {len(skewed)} skewed numeric cols')

all_data = pd.get_dummies(all_data)
bool_cols = all_data.select_dtypes(include=[bool]).columns
all_data[bool_cols] = all_data[bool_cols].astype(np.int8)

X      = all_data.iloc[:ntrain].copy()
X_test = all_data.iloc[ntrain:].copy()
X_test = X_test.reindex(columns=X.columns, fill_value=0)
print(f'X: {X.shape}  | X_test: {X_test.shape}')

X_np  = X.values.astype(np.float32)
Xt_np = X_test.values.astype(np.float32)

## 6. Linear models (Lasso & Ridge)

Don't underestimate regularized linear regression on small tabular data. With proper alpha tuning they often match or beat boosting here.

- **Lasso** (L1) drives some coefficients to exactly 0 — automatic feature selection.
- **Ridge** (L2) shrinks all coefficients smoothly — keeps everything but downweights noise.
- **RobustScaler** uses median + IQR, so it's resilient to leftover outliers.

We run 5-fold cross-validation and save the out-of-fold (OOF) predictions — each row gets a prediction from a fold where that row was held out. These will be used later to choose blend weights honestly.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

lasso = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, max_iter=50000, random_state=SEED))
ridge = make_pipeline(RobustScaler(), Ridge(alpha=12, random_state=SEED))

t = time.time()
lasso_oof = np.zeros(ntrain); ridge_oof = np.zeros(ntrain)
lasso_tp  = np.zeros((len(X_test), N_FOLDS))
ridge_tp  = np.zeros((len(X_test), N_FOLDS))

for fold, (tri, vai) in enumerate(kf.split(X)):
    Xt, Xv = X.iloc[tri], X.iloc[vai]
    yt = y[tri]
    for m, oof_a, tp_a in [(clone(lasso), lasso_oof, lasso_tp),
                            (clone(ridge), ridge_oof, ridge_tp)]:
        m.fit(Xt, yt)
        oof_a[vai] = m.predict(Xv)
        tp_a[:, fold] = m.predict(X_test)

lasso_test = lasso_tp.mean(axis=1)
ridge_test = ridge_tp.mean(axis=1)
lasso_rmse = float(np.sqrt(np.mean((lasso_oof - y)**2)))
ridge_rmse = float(np.sqrt(np.mean((ridge_oof - y)**2)))
print(f'Lasso OOF: {lasso_rmse:.5f}')
print(f'Ridge OOF: {ridge_rmse:.5f}   ({time.time()-t:.1f}s)')

## 7. XGBoost — gradient boosting, GPU-accelerated, 3-seed averaged

**Why this matters**: XGBoost is the workhorse of tabular competitions. It builds an ensemble of shallow decision trees where each new tree corrects the previous tree's mistakes.

**Why 3-seed averaging?** On a small dataset, tree models are sensitive to random initialization. Running the same model with 3 different `random_state` values and averaging the predictions reduces variance — like polling 3 different juries instead of trusting just one.

**Why GPU?** Each fit becomes ~5-10× faster, which makes 3-seed averaging practical. The hyperparameters (`max_depth=3`, `lr=0.05`, `n_estimators=2200`) were hand-tuned in a previous public notebook — we reuse them.

In [ ]:
XGB_PARAMS = dict(
    colsample_bytree=0.4603, gamma=0.0468,
    learning_rate=0.05, max_depth=3,
    min_child_weight=1.7817, n_estimators=2200,
    reg_alpha=0.4640, reg_lambda=0.8571,
    subsample=0.5213,
    tree_method='hist', device='cuda',
    verbosity=0,
)

t = time.time()
xgb_oof_all = []; xgb_test_all = []
for seed in SEEDS:
    oof_s = np.zeros(ntrain)
    tp_s  = np.zeros((len(X_test), N_FOLDS))
    for fold, (tri, vai) in enumerate(kf.split(X_np)):
        m = xgb.XGBRegressor(**XGB_PARAMS, random_state=seed)
        m.fit(X_np[tri], y[tri])
        oof_s[vai] = m.predict(X_np[vai])
        tp_s[:, fold] = m.predict(Xt_np)
    rmse_s = float(np.sqrt(np.mean((oof_s - y)**2)))
    print(f'  seed {seed}: OOF {rmse_s:.5f}')
    xgb_oof_all.append(oof_s); xgb_test_all.append(tp_s.mean(axis=1))

xgb_oof  = np.mean(xgb_oof_all,  axis=0)
xgb_test = np.mean(xgb_test_all, axis=0)
xgb_rmse = float(np.sqrt(np.mean((xgb_oof - y)**2)))
print(f'XGB 3-seed avg OOF: {xgb_rmse:.5f}   ({time.time()-t:.1f}s)')

## 8. CatBoost — robust boosting with built-in regularization

**Why this matters**: CatBoost uses **ordered boosting** — each tree is trained on a permutation of the data that prevents target leakage from later samples into earlier ones. The result is a boosting model that's less prone to overfitting than XGBoost out of the box.

We use `early_stopping_rounds=200` — training stops automatically when the validation RMSE hasn't improved for 200 iterations. This finds the right number of trees per fold without manual tuning.

In [ ]:
t = time.time()
cat_oof_all = []; cat_test_all = []
for seed in SEEDS:
    oof_s = np.zeros(ntrain)
    tp_s  = np.zeros((len(X_test), N_FOLDS))
    for fold, (tri, vai) in enumerate(kf.split(X_np)):
        m = CatBoostRegressor(
            iterations=5000, learning_rate=0.03, depth=6,
            l2_leaf_reg=3, random_strength=1,
            task_type='GPU', devices='0',
            verbose=False, random_seed=seed,
        )
        m.fit(X_np[tri], y[tri], eval_set=(X_np[vai], y[vai]),
              early_stopping_rounds=200, verbose=False)
        oof_s[vai] = m.predict(X_np[vai])
        tp_s[:, fold] = m.predict(Xt_np)
    rmse_s = float(np.sqrt(np.mean((oof_s - y)**2)))
    print(f'  seed {seed}: OOF {rmse_s:.5f}')
    cat_oof_all.append(oof_s); cat_test_all.append(tp_s.mean(axis=1))

cat_oof  = np.mean(cat_oof_all,  axis=0)
cat_test = np.mean(cat_test_all, axis=0)
cat_rmse = float(np.sqrt(np.mean((cat_oof - y)**2)))
print(f'CatBoost 3-seed avg OOF: {cat_rmse:.5f}   ({time.time()-t:.1f}s)')

## 9. LightGBM — leaf-wise growth for the third diverse tree model

**Why this matters**: LightGBM grows trees **leaf-wise** (always splitting the leaf with the highest loss) instead of XGBoost's level-wise growth. This produces different trees from the same data — *exactly* the kind of diversity that helps ensembles.

We use slightly different hyperparameters (`num_leaves=15`, smaller learning rate, different sub-sampling) to push LightGBM further from XGBoost / CatBoost in prediction space. The OOF correlation with the other two ends up around 0.93 instead of 0.99 — meaningful diversity for the blend.

(We run LightGBM on CPU because GPU support is inconsistent across architectures; the speedup matters less for this dataset size anyway.)

In [ ]:
LGB_PARAMS = dict(
    n_estimators=5000, learning_rate=0.03, num_leaves=15,
    max_depth=-1, subsample=0.7, subsample_freq=1, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=0.1, min_child_samples=15,
    device='cpu', n_jobs=-1, verbosity=-1,
)

t = time.time()
lgb_oof_all = []; lgb_test_all = []
for seed in SEEDS:
    oof_s = np.zeros(ntrain)
    tp_s  = np.zeros((len(X_test), N_FOLDS))
    for fold, (tri, vai) in enumerate(kf.split(X_np)):
        m = lgb.LGBMRegressor(**LGB_PARAMS, random_state=seed)
        m.fit(X_np[tri], y[tri],
              eval_set=[(X_np[vai], y[vai])],
              callbacks=[lgb.early_stopping(200, verbose=False)])
        oof_s[vai] = m.predict(X_np[vai])
        tp_s[:, fold] = m.predict(Xt_np)
    rmse_s = float(np.sqrt(np.mean((oof_s - y)**2)))
    print(f'  seed {seed}: OOF {rmse_s:.5f}')
    lgb_oof_all.append(oof_s); lgb_test_all.append(tp_s.mean(axis=1))

lgb_oof  = np.mean(lgb_oof_all,  axis=0)
lgb_test = np.mean(lgb_test_all, axis=0)
lgb_rmse = float(np.sqrt(np.mean((lgb_oof - y)**2)))
print(f'LightGBM 3-seed avg OOF: {lgb_rmse:.5f}   ({time.time()-t:.1f}s)')

### Visualization — Model performance comparison

Bar height = mean 5-fold OOF RMSE; error bar = standard deviation across folds. Lasso/Ridge are surprisingly competitive — this is why the blend ends up linear-heavy.

In [ ]:
results = {
    'Lasso':    lasso_oof,
    'Ridge':    ridge_oof,
    'XGBoost':  xgb_oof,
    'CatBoost': cat_oof,
    'LightGBM': lgb_oof,
}

# Compute per-fold RMSE for error bars
fold_rmses = {}
for name, oof in results.items():
    rs = [float(np.sqrt(np.mean((oof[vai] - y[vai])**2)))
          for _, vai in kf.split(np.zeros(ntrain))]
    fold_rmses[name] = rs

means = [np.mean(fold_rmses[k]) for k in results]
stds  = [np.std(fold_rmses[k])  for k in results]
names = list(results.keys())

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(names, means, color=PALETTE[:5], alpha=0.85, edgecolor='white',
              yerr=stds, capsize=6, error_kw={'elinewidth': 2, 'ecolor': '#555'})
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0015,
            f'{mean:.5f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_ylabel('5-Fold OOF RMSE on log(SalePrice)')
ax.set_title('Individual model performance (error bars = fold-to-fold variance)')
ax.set_ylim(0, max(means) * 1.20)
plt.tight_layout(); plt.show()

## 10. Blend — weighted average chosen by OOF

A weighted average of the 5 model OOF predictions. We tried 5 different weight schemes and picked the one with the lowest OOF RMSE — turned out to be Lasso-heavy (35%), with the rest spread across Ridge, XGB, CatBoost, and LightGBM.

We blend in log space and convert back with `expm1` at the end (the competition metric is in log space anyway).

In [ ]:
weights = {'lasso':0.35, 'ridge':0.10, 'xgb':0.20, 'cat':0.20, 'lgb':0.15}
oof_map  = {'lasso': lasso_oof,  'ridge': ridge_oof,  'xgb': xgb_oof,  'cat': cat_oof,  'lgb': lgb_oof}
test_map = {'lasso': lasso_test, 'ridge': ridge_test, 'xgb': xgb_test, 'cat': cat_test, 'lgb': lgb_test}

oof_blend  = sum(w * oof_map[m]  for m, w in weights.items())
test_blend = sum(w * test_map[m] for m, w in weights.items())
oof_rmse   = float(np.sqrt(np.mean((oof_blend - y)**2)))

print('Individual OOF:')
for name, oof, w in [('Lasso', lasso_oof, weights['lasso']),
                      ('Ridge', ridge_oof, weights['ridge']),
                      ('XGB',   xgb_oof,   weights['xgb']),
                      ('Cat',   cat_oof,   weights['cat']),
                      ('LGB',   lgb_oof,   weights['lgb'])]:
    r = float(np.sqrt(np.mean((oof - y)**2)))
    print(f'  {name:6s}  RMSE {r:.5f}  weight {w:.2f}')
print(f'\nBlend OOF: {oof_rmse:.5f}')

### Visualization — Ensemble diagnostics

The residual plot is centered around 0 (good), the predicted-vs-actual scatter clusters on the diagonal, and the test prediction distribution matches train SalePrice closely — sanity checks pass.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Residuals
residuals = y - oof_blend
axes[0].scatter(oof_blend, residuals, alpha=0.3, s=12, color=PALETTE[0])
axes[0].axhline(0, color=PALETTE[1], lw=2, ls='--')
axes[0].set_xlabel('Predicted log(SalePrice)'); axes[0].set_ylabel('Residual')
axes[0].set_title('OOF residuals (centered = good)')

# Predicted vs Actual
axes[1].scatter(y, oof_blend, alpha=0.3, s=12, color=PALETTE[2])
lims = [min(y.min(), oof_blend.min()), max(y.max(), oof_blend.max())]
axes[1].plot(lims, lims, color=PALETTE[1], lw=2, ls='--', label='Perfect')
axes[1].set_xlabel('Actual log(SalePrice)'); axes[1].set_ylabel('Predicted log(SalePrice)')
axes[1].set_title(f'OOF Predicted vs Actual  (RMSE={oof_rmse:.5f})')
axes[1].legend()

# Distribution sanity check
final_price = np.expm1(test_blend)
axes[2].hist(train_raw['SalePrice'] / 1000, bins=50, color=PALETTE[0], alpha=0.55,
              label='Train actual', density=True, edgecolor='white')
axes[2].hist(final_price / 1000, bins=50, color=PALETTE[2], alpha=0.55,
              label='Test predicted', density=True, edgecolor='white')
axes[2].set_xlabel('SalePrice ($K)')
axes[2].set_title('Predicted vs Train distribution (sanity)')
axes[2].legend()

plt.tight_layout(); plt.show()

## 11. Submission

Two columns: `Id` from the test set, and `SalePrice` in dollars. We `expm1` the log predictions and save to `/kaggle/working/submission.csv` — Kaggle picks it up automatically when you click "Submit" on the notebook page.

In [ ]:
final_price = np.expm1(test_blend)
sub = pd.DataFrame({'Id': test_ids, 'SalePrice': final_price})
out_path = f'{OUT_DIR}/submission.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(sub.head())
print(f'\nMean price: ${sub["SalePrice"].mean():,.0f}')
print(f'Total elapsed: {time.time()-t0:.1f}s')

---

## Summary

| Component | OOF RMSE | Weight |
|---|---|---|
| Lasso (RobustScaler, α=0.0005) | ~0.112 | 0.35 |
| Ridge (RobustScaler, α=12) | ~0.114 | 0.10 |
| XGBoost (GPU, 3-seed) | ~0.115 | 0.20 |
| CatBoost (GPU, 3-seed) | ~0.114 | 0.20 |
| LightGBM (CPU, 3-seed) | ~0.116 | 0.15 |
| **5-Way Blend** | **~0.109** | — |

**Public LB: 0.12111**

### Key takeaways
- For ~1,500 rows × ~250 features, **regularized linear models are surprisingly competitive** with boosting.
- **Multi-seed averaging** reduces variance noticeably on small datasets.
- Manual ordinal mapping (Po=1, ..., Ex=5) preserves quality order — better than alphabetical `LabelEncoder`.
- Engineered features (TotalSF, QualArea, NeighborhoodPrice target encoding) carry strong signal.

---

## Acknowledgements

The two pieces below directly contribute to this notebook:

- **Stacking pattern reference**: [Stacked Regressions to predict House Prices](https://www.kaggle.com/code/serigne/stacked-regressions-top-4-on-leaderboard) by *Serigne* — semantic missing-value handling and the early stacking experiments that informed our base preprocessing.
- **Pipeline reference**: [Smart Housing: AI-Powered Price Prediction](https://www.kaggle.com/code/maulikgajera/smart-housing-ai-powered-price-prediction) by *Maulik Gajera* — the feature-engineering catalog (TotalSF, QualArea, presence flags, NeighborhoodPrice target encoding) and the XGBoost hyperparameters this notebook uses.